In [1]:
import json
import pyomo.environ as pyo
import numpy as np

In [2]:
with open("../../data/expected_returns_covariance_matrix.json") as f:
    data = json.load(f)
    tickers = data["tickers"] #list
    exp_ret = data["expected_returns"] #list

with open("../../data/scenario_cumulative_returns.json") as f:
    scenarios = json.load(f) #nested dict 
    scenarios['0']['TSM'] # example of how to access an entry

In [3]:

exp_ret_annual = [r**52 for r in exp_ret]


In [4]:
scenarios['0']['TSM']

1.3833857691851517

In [5]:
help(pyo.Var)

Help on class Var in module pyomo.core.base.var:

class Var(pyomo.core.base.indexed_component.IndexedComponent, pyomo.core.base.indexed_component.IndexedComponent_NDArrayMixin)
 |  Var(*args, **kwargs)
 |
 |  A numeric variable, which may be defined over an index.
 |
 |  Args:
 |      domain (Set or function, optional): A Set that defines valid
 |          values for the variable (e.g., ``Reals``, ``NonNegativeReals``,
 |          ``Binary``), or a rule that returns Sets.  Defaults to ``Reals``.
 |      within (Set or function, optional): An alias for ``domain``.
 |      bounds (tuple or function, optional): A tuple of ``(lower, upper)``
 |          bounds for the variable, or a rule that returns tuples.
 |          Defaults to ``(None, None)``.
 |      initialize (float or function, optional): The initial value for
 |          the variable, or a rule that returns initial values.
 |      rule (float or function, optional): An alias for ``initialize``.
 |      dense (bool, optional): In

In [6]:
def solve_cvar(alpha, scenarios, tickers, expected_returns, target_return, K=None,
               tolerance=0.0001, time_limit=10, solver_options=None):
    """
    Minimize Conditional Value-at-Risk (CVaR) of portfolio returns.
    
    Args:
        alpha: CVaR confidence level (e.g., 0.95 for 95% CVaR)
        scenarios: Dict of scenarios with cumulative returns, keyed by scenario ID
                   e.g., {'0': {'TICKER': multiplier, ...}, '1': {...}, ...}
        tickers: List of ticker symbols
        expected_returns: List of expected returns (geometric multipliers, e.g., 1.02)
        target_return: Target portfolio expected return (geometric multiplier)
        K: Maximum number of stocks in portfolio (None = no cardinality constraint)
        tolerance: Tolerance band around target return (default 0.0001)
        time_limit: Solver time limit in seconds (default 10)
        solver_options: Optional dict of additional solver options
    
    Returns:
        Tuple: (weights_dict, gamma_value, cvar_objective, solver_result)
    """
    S = len(scenarios)
    n = len(tickers)
    M = 1.0  # Big-M constant
    
    # Create model
    model = pyo.ConcreteModel()
    
    # Decision variables (non-negativity and binary domain enforced via variable domains)
    model.w = pyo.Var(range(n), domain=pyo.NonNegativeReals)  # Portfolio weights
    model.gamma = pyo.Var(domain=pyo.Reals)  # CVaR threshold
    model.x = pyo.Var(range(n), domain=pyo.Binary)  # Stock selection
    model.z = pyo.Var(range(S), domain=pyo.NonNegativeReals)  # Loss overage
    
    # Objective: minimize CVaR
    coeff = 1.0 / ((1 - alpha) * S)
    model.obj = pyo.Objective(
        expr=model.gamma + coeff * sum(model.z[s] for s in range(S)),
        sense=pyo.minimize
    )
    
    # Budget constraint
    model.budget = pyo.Constraint(
        expr=sum(model.w[i] for i in range(n)) == 1
    )
    
    # Target return lower bound
    model.target_lower = pyo.Constraint(
        expr=sum(model.w[i] * expected_returns[i] for i in range(n)) >= target_return - tolerance
    )
    
    # Target return upper bound
    model.target_upper = pyo.Constraint(
        expr=sum(model.w[i] * expected_returns[i] for i in range(n)) <= target_return + tolerance
    )
    
    # Loss constraint
    model.loss_constraint = pyo.ConstraintList()
    scenario_keys = list(scenarios.keys())
    for s_idx, s_key in enumerate(scenario_keys):
        loss_expr = 1 - sum(model.w[i] * scenarios[s_key][tickers[i]] for i in range(n))
        model.loss_constraint.add(model.z[s_idx] >= loss_expr - model.gamma)
    
    # Cardinality constraint and Big-M linkage (only if K is specified)
    if K is not None:
        # Cardinality constraint
        model.cardinality = pyo.Constraint(
            expr=sum(model.x[i] for i in range(n)) <= K
        )
        
        # Big-M linkage
        model.big_m = pyo.ConstraintList()
        for i in range(n):
            model.big_m.add(model.w[i] <= M * model.x[i])
    
    # Create solver
    solver = pyo.SolverFactory("highs")
    solver.options['time_limit'] = time_limit
    
    # Apply user-provided solver options
    if solver_options:
        for key, value in solver_options.items():
            solver.options[key] = value
    
    # Solve
    result = solver.solve(model)
    
    # Check for timeout
    if result.solver.termination_condition == pyo.TerminationCondition.maxTimeLimit:
        print(f"WARNING: Solver timeout at target_return={target_return}")
    
    # Extract results
    weights = {tickers[i]: pyo.value(model.w[i]) for i in range(n)}
    gamma_value = pyo.value(model.gamma)
    cvar_objective = pyo.value(model.obj)
    
    return weights, gamma_value, cvar_objective, result

In [7]:
# Example with cardinality constraint: at most 20 stocks in portfolio
K = 30

weights_constrained, gamma_c, cvar_obj_c, result_c = solve_cvar(
    alpha=0.95,
    scenarios=scenarios,
    tickers=tickers,
    expected_returns=exp_ret_annual,
    target_return=1.3,
    K=K,
    tolerance=0.1,
    time_limit=10
)

print(f"Optimization Result: {result_c.solver.termination_condition}")
print(f"CVaR Objective Value: {cvar_obj_c:.6f}")
print(f"Number of stocks selected: {sum(1 for w in weights_constrained.values() if w > 1e-6)}")
print(f"\nPortfolio Weights (top 10):") 
sorted_weights_c = sorted(weights_constrained.items(), key=lambda x: x[1], reverse=True)
for ticker, weight in sorted_weights_c[:10]:
    if weight > 1e-6:
        print(f"  {ticker}: {weight:.4f}")

Optimization Result: optimal
CVaR Objective Value: 0.046575
Number of stocks selected: 12

Portfolio Weights (top 10):
  LOTB.BR: 0.2233
  RHM.DE: 0.1724
  LLY: 0.1168
  NVDA: 0.0918
  ELI.BR: 0.0818
  005930.KS: 0.0730
  SUNPHARMA.NS: 0.0725
  9983.T: 0.0615
  NEE: 0.0440
  MCK: 0.0351


In [8]:
#### Efficient frontier sweep (CVaR with cardinality constraint)

alpha = 0.95
K = 30
increment = 0.01
tolerance = 0.001
time_limit = 10

min_mu = 1.0  # 0% growth
max_mu = max(exp_ret_annual)
# arange excludes the endpoint, so append max_mu to make sure the top of the range is solved
range_mu = np.append(np.arange(min_mu, max_mu, increment), max_mu).tolist()

results = []

for given_mu in range_mu:
    weights, gamma, cvar_obj, result = solve_cvar(
        alpha=alpha,
        scenarios=scenarios,
        tickers=tickers,
        expected_returns=exp_ret_annual,
        target_return=given_mu,
        K=K,
        tolerance=tolerance,
        time_limit=time_limit,
    )
    # Fail loudly: only optimal solves and timeouts (recorded with their status) are allowed
    termination = result.solver.termination_condition
    if termination not in (pyo.TerminationCondition.optimal, pyo.TerminationCondition.maxTimeLimit):
        raise RuntimeError(f"Solve failed at target_return={given_mu}: {termination}")
    results.append({
        "target_return": given_mu,
        "weights": weights,
        "cvar": cvar_obj,
        "gamma": gamma,
        "solver_status": str(result.solver.status),
        "termination_condition": str(termination),
    })
    print(f"Solved target_return={given_mu:.4f}")

frontier_output = {
    "results": results,
    "summary": {
        "total_solves": len(results),
        "alpha": alpha,
        "K": K,
        "increment": increment,
        "tolerance": tolerance,
    },
}

with open("../../data/solver.output.json", "w") as f:
    json.dump(frontier_output, f, indent=2)

print("Results written to ../../data/solver.output.json")


Solved target_return=1.0000
Solved target_return=1.0100
Solved target_return=1.0200
Solved target_return=1.0300
Solved target_return=1.0400
Solved target_return=1.0500
Solved target_return=1.0600
Solved target_return=1.0700
Solved target_return=1.0800
Solved target_return=1.0900
Solved target_return=1.1000
Solved target_return=1.1100
Solved target_return=1.1200
Solved target_return=1.1300
Solved target_return=1.1400
Solved target_return=1.1500
Solved target_return=1.1600
Solved target_return=1.1700
Solved target_return=1.1800
Solved target_return=1.1900
Solved target_return=1.2000
Solved target_return=1.2100
Solved target_return=1.2200
Solved target_return=1.2300
Solved target_return=1.2400
Solved target_return=1.2500
Solved target_return=1.2600
Solved target_return=1.2700
Solved target_return=1.2800
Solved target_return=1.2900
Solved target_return=1.3000
Solved target_return=1.3100
Solved target_return=1.3200
Solved target_return=1.3300
Solved target_return=1.3400
Solved target_return